# Analyse Agriculture through Time

Read cropping areas and land-cover classes for one micro-watershed from 2017–18 to 2024–25.

Run the cells in order. Each step uses data from the previous cells. You can edit the place, identifier, columns and chart settings as you go.


## Set up Python

Run these two collapsed cells once. They load the libraries and starting location. Expand them to see or change the setup.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import ast
import json
from getpass import getpass
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
GEOSERVER = 'https://geoserver.core-stack.org:8443/geoserver/'
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


## Choose the tehsil

The starting place is Hilsa, Nalanda, Bihar. A notebook downloaded from GeoLibre uses the selected tehsil. Change these names to read another place.


In [ ]:
state = SCOPE["state"].lower().replace(" ", "_")
district = SCOPE["district"].lower().replace(" ", "_")
tehsil = SCOPE["tehsil"].lower().replace(" ", "_")
place = {"state": state, "district": district, "tehsil": tehsil}
place


## The layers we will use

Each link reads a vector layer as GeoJSON. GeoJSON contains a feature list; each feature has a shape and its data fields.


In [ ]:
layer_urls = {
    "cropping": f"{GEOSERVER}crop_intensity/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=crop_intensity:{district}_{tehsil}_intensity&outputFormat=application/json&srsName=EPSG:4326",
    "land_cover": f"{GEOSERVER}lulc_vector/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=lulc_vector:lulc_vector_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
}
pd.DataFrame(layer_urls.items(), columns=["Layer", "GeoJSON URL"])


## Read the cropping-intensity layer

The area fields are hectares. Their year suffix is the starting year of the July–June period.


In [ ]:
cropping_response = requests.get(layer_urls["cropping"], timeout=90)
cropping_response.raise_for_status()
cropping_geojson = cropping_response.json()
cropping = gpd.GeoDataFrame.from_features(cropping_geojson["features"], crs="EPSG:4326")
cropping.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_cropping_intensity_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['uid', 'cropping_intensity_2017', 'single_kharif_cropped_area_2017', 'doubly_cropped_area_2017']), ["name", "type", "description"]]


## Choose one micro-watershed

`uid` is the MWS identifier in this layer. Start with the first one, or replace `mws_id` with another identifier from the displayed list.


In [ ]:
mws_ids = cropping["uid"].sort_values().tolist()
display(pd.DataFrame({"MWS identifier": mws_ids}))
mws_id = mws_ids[0]
mws_id


## List the years

Read the published cropping-intensity field for each requested year.


In [ ]:
crop_row = cropping.loc[cropping["uid"] == mws_id].iloc[0]
pd.DataFrame({"Year": [f"{y}–{y+1}" for y in YEARS],
              "Cropping intensity": [crop_row[f"cropping_intensity_{y}"] for y in YEARS]})


## Area under each cropping category

The four categories separate single Kharif, single non-Kharif, double and triple cropping. Edit the selected columns to explore them.


In [ ]:
crop_area = pd.DataFrame({
    "Single Kharif": [crop_row[f"single_kharif_cropped_area_{y}"] for y in YEARS],
    "Single non-Kharif": [crop_row[f"single_non_kharif_cropped_area_{y}"] for y in YEARS],
    "Double cropping": [crop_row[f"doubly_cropped_area_{y}"] for y in YEARS],
    "Triple cropping": [crop_row[f"triply_cropped_area_{y}"] for y in YEARS]
}, index=[f"{y}–{y+1}" for y in YEARS])
crop_area.index.name = "Year"
crop_area


## Uncropped or fallow land

Show these explicitly named fields when supplied. Do not infer fallow land by subtracting the four categories from the MWS area.


In [ ]:
uncropped_fields = [f"uncropped_area_{y}" for y in YEARS]
fallow_fields = [f"fallow_area_{y}" for y in YEARS]
available_fields = cropping.columns.intersection(uncropped_fields + fallow_fields)
cropping.loc[cropping["uid"] == mws_id, available_fields].T


## Plot the hectare values

The stacked columns show the recorded area in each category.


In [ ]:
crop_area.plot.bar(stacked=True, figsize=(10, 4), ylabel="Area (ha)")
plt.legend(title="Cropping category", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


## Show the cropping shares

Divide each category by the sum of the four recorded cropping areas for that year. These are shares of recorded cropped land, not of the whole MWS.


In [ ]:
cropped_total = crop_area.sum(axis=1, min_count=4).replace(0, float("nan"))
crop_share = crop_area.div(cropped_total, axis=0) * 100
display(crop_share)
crop_share.plot.bar(stacked=True, figsize=(10, 4), ylabel="Share of recorded cropped area (%)")
plt.legend(title="Cropping category", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


## Read the land-cover summary

This vector layer records the areas of broader land-cover classes and cropping classes for each year.


In [ ]:
land_cover_response = requests.get(layer_urls["land_cover"], timeout=90)
land_cover_response.raise_for_status()
land_cover_geojson = land_cover_response.json()
land_cover = gpd.GeoDataFrame.from_features(land_cover_geojson["features"], crs="EPSG:4326")
land_cover.drop(columns="geometry").head()


## Read broader land-cover areas

Read the selected MWS using the exact field prefixes below. The values are hectares.


In [ ]:
cover_row = land_cover.loc[land_cover["uid"] == mws_id].iloc[0]
cover_area = pd.DataFrame({
    "Built-up": [cover_row[f"built-up_area_{y}"] for y in YEARS],
    "Trees and forest": [cover_row[f"tree_forest_area_{y}"] for y in YEARS],
    "Shrubs and scrub": [cover_row[f"shrub_scrub_area_{y}"] for y in YEARS],
    "Barren land": [cover_row[f"barrenlands_area_{y}"] for y in YEARS],
    "Cropland": [cover_row[f"cropland_area_{y}"] for y in YEARS]
}, index=crop_area.index)
cover_area


## Plot the land-cover areas

Each line follows one published land-cover class. Cropping-category areas remain in the table above.


In [ ]:
cover_area.plot(marker="o", figsize=(10, 4), ylabel="Area (ha)", xlabel="Year")
plt.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


## Connect to the CoRE Stack API

Read endpoint specifications at [api-doc.core-stack.org](https://api-doc.core-stack.org). The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains how to register, generate an API key and use it. The key goes in the `X-API-Key` header. This cell reads `CORE_STACK_API_KEY` from your environment, or asks for it without showing it. The key is not written into the notebook. After each API request, run the collapsed parsing cell: it keeps the raw response text and reads it with `json.loads`, which accepts `NaN` as a missing numeric value. Expand the cell to inspect the code.


In [ ]:
from inspect import isawaitable

api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key

api_headers = {"X-API-Key": str(api_key).strip()}


## Read the tehsil tables from the API

`get_tehsil_data` returns a dictionary of tables for the same place. Here we make the request once and reuse the returned tables below.


In [ ]:
api_response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_response.raise_for_status()


In [ ]:
raw_api_data_string = api_response.text
api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))


In [ ]:
api_data = api_payload
pd.DataFrame({"Table": api_data.keys(), "Rows": [len(rows) for rows in api_data.values()]})


## Cropping areas from the API

The API table is named `croppingIntensity_annual`. Its area fields include `in_ha` and the full year range.


In [ ]:
api_crop_rows = pd.DataFrame(api_data["croppingIntensity_annual"])
api_crop_row = api_crop_rows.loc[api_crop_rows["uid"] == mws_id].iloc[0]
api_crop_area = pd.DataFrame({
    "Single Kharif": [api_crop_row[f"single_kharif_cropped_area_in_ha_{y}-{y+1}"] for y in YEARS],
    "Single non-Kharif": [api_crop_row[f"single_non_kharif_cropped_area_in_ha_{y}-{y+1}"] for y in YEARS],
    "Double cropping": [api_crop_row[f"doubly_cropped_area_in_ha_{y}-{y+1}"] for y in YEARS],
    "Triple cropping": [api_crop_row[f"triply_cropped_area_in_ha_{y}-{y+1}"] for y in YEARS]
}, index=crop_area.index)
api_crop_area


## Land-cover areas from the API

Read the corresponding hectare fields in `lulc_vector`.


In [ ]:
api_cover_rows = pd.DataFrame(api_data["lulc_vector"])
api_cover_row = api_cover_rows.loc[api_cover_rows["uid"] == mws_id].iloc[0]
api_cover_area = pd.DataFrame({
    "Built-up": [api_cover_row[f"built-up_area_in_ha_{y}"] for y in YEARS],
    "Trees and forest": [api_cover_row[f"tree_forest_area_in_ha_{y}"] for y in YEARS],
    "Shrubs and scrub": [api_cover_row[f"shrub_scrub_area_in_ha_{y}"] for y in YEARS],
    "Barren land": [api_cover_row[f"barrenlands_area_in_ha_{y}"] for y in YEARS],
    "Cropland": [api_cover_row[f"cropland_area_in_ha_{y}"] for y in YEARS]
}, index=crop_area.index)
api_cover_area
